# Pooled regions 1: sites and function-CAV fits for every MPRA region, five models

Meeting 2026-10-27, ask 1: evaluate on **all** MPRA regions at once instead of one region at a time. So far only
regions 0, 8 and 20 (HepG2) had hits, sites and fits. This notebook builds them for **every region in the MPRA file**
(21 regions) and fits the function direction for **every TF with a motif hit in any region**, on five models.
`Pooled Regions 2 - MPRA Evaluation over Every Region.ipynb` then does the evaluation (CPU).

**What changes compared with the per-region notebooks**

* **One fit per TF, shared by all regions.** Before, each region had its own copy of a TF's sites, with only the
  windows overlapping *that* region removed. Here a TF's sites drop every window that overlaps **any** of the MPRA
  regions, and the direction is fitted once. Every hit of that TF, in whatever region, gets its out-of-fold P(bound)
  from the same fit. Sites already built for regions 0/8/20 are reused (minus the newly overlapping pairs), so only new
  TFs go to ENCODE.
* **Only the layer each model used before** (NT-v2 500M L24, DNABERT-2 L4, GENA-LM L8 and L11, HyenaDNA L3, NT 2.5B L14).
  Each layer's fit is independent of the others, so the result at that layer is the same as fitting all layers, at
  a fraction of the cost (fitting all layers was ~90% of the run time). No layer curve is produced here.
* Everything else (site labels, PWM/GC/CpG matching, the readout, the L2-logistic estimator, 5 outer folds) is copied
  unchanged from `upstream/Function CAV - Strict Labels, PWM-GC-CpG-Matched L2 Logistic, Region 20.ipynb` and
  `Function CAV - Cross-Model, Region 0.ipynb`.

**Cell types.** The CAV is trained on HepG2 + K562 ChIP. Only regions 0, 8, 20 (HepG2) and 1 (K562) were measured in
those cells; the others in HEK293T, HeLa, HEL, Neuro-2a, MIN6, etc. They are all built here; the evaluation reports
them separately.

**Run order**
1. Section 1 on a **CPU** runtime (FIMO, ENCODE downloads, site building; a few hours the first time, then cached).
2. Section 2 on a **GPU** runtime (A100). Resumable per (model, TF): a disconnected session continues where it
   stopped. `MODELS` in 2.1 can be cut down to run one model per session. TFs are done in order of how many hits they
   have, so a partial run already covers most hits.

Output on Drive: `motif_token_study/fcav_pool/` (`regions/r<k>/`, `sites/`, `mpra_windows.json`, `fits/<model>/`).

---
# Section 1 — every region, hits, ENCODE, sites *(CPU runtime)*

In [ ]:
!pip install -q pymemesuite pyfaidx requests
from google.colab import drive
drive.mount('/content/drive')

## 1.1 Settings and every MPRA region

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, shutil
import numpy as np, pandas as pd

ACCESSORY = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/accessary_files"
MPRA_PATH = f"{ACCESSORY}/mpra_sequences_with_embeddings_nt.pkl"   # a .csv with the same columns also works
REGIONS = None                # None = every region in the MPRA file; or a list such as [0, 1, 8, 20]
CELLS = ["HepG2", "K562"]     # ChIP-seq and DNase from these cells, sites pooled
N_PER_CLASS = 2000            # at most this many sites per class after matching
MATCH_VARS = ["pwm", "gc", "cpg_oe"]
N_MATCH_BINS = 5              # quantile bins per matching variable (5 x 5 x 5 joint bins)
UNBOUND_POOL = 10             # unbound candidates collected: UNBOUND_POOL x N_PER_CLASS, before matching
BOUND_TOP = 0.25              # bound: peaks in the top quarter by signalValue within each experiment
UNBOUND_MIN_DIST = 1000       # unbound: at least this far from every peak of the TF, relaxed peaks included

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL = f"{WORK_DIR}/fcav_pool"
SHARE = [f"{WORK_DIR}/{d}/sites" for d in ("fcav_pgc_hk", "fcav_pgc_hk_r0", "fcav_pgc_hk_r20")]   # sites built before
for d in ("", "/sites", "/regions"):
    os.makedirs(POOL + d, exist_ok=True)
json.dump({"regions": REGIONS, "cells": CELLS, "n_per_class": N_PER_CLASS, "match_vars": MATCH_VARS,
           "n_match_bins": N_MATCH_BINS, "unbound_pool": UNBOUND_POOL, "bound_top": BOUND_TOP,
           "unbound_min_dist": UNBOUND_MIN_DIST}, open(f"{POOL}/settings.json", "w"), indent=1)
TOK, MOFF, WLEN = C["tok"], C["motif_offset"], C["window_len"]
pd.set_option("display.width", 250)
_RC = str.maketrans("ACGT", "TGCA")


def revcomp(s):
    return s.translate(_RC)[::-1]


_m = pd.read_csv(MPRA_PATH, low_memory=False) if MPRA_PATH.endswith(".csv") else pd.read_pickle(MPRA_PATH)
_m = _m.drop(columns=[c for c in ("wt_embedding", "mut_embedding", "sequence_mut") if c in _m.columns])
_m["region_id"] = _m["region_id"].astype(int)
if REGIONS is not None:
    _m = _m[_m["region_id"].isin(REGIONS)]
SEQ, CELL, HINT, SNV0 = {}, {}, {}, {}
for rid, g in _m.groupby("region_id"):
    s = g["sequence"].dropna().astype(str).str.upper().unique()
    assert len(s) == 1, f"region {rid}: {len(s)} different WT sequences"
    SEQ[rid] = s[0]
    CELL[rid] = ",".join(sorted(g["cellline"].astype(str).unique())) if "cellline" in g.columns else "?"
    HINT[rid] = str(g["chr"].iloc[0]) if "chr" in g.columns else None
    v = g[(g["alt"] != "-") & g["log2FC"].notna()].copy()
    v["local"] = v["pos"].astype(int) - v["start"].astype(int)
    v["ref"], v["alt"] = v["ref"].astype(str).str.upper(), v["alt"].astype(str).str.upper()
    assert all(0 <= p < len(SEQ[rid]) and SEQ[rid][p] == r for p, r in zip(v["local"], v["ref"])), f"region {rid}: ref mismatch"
    SNV0[rid] = v.groupby(["local", "ref", "alt"]).agg(log2FC=("log2FC", "mean"), n_meas=("log2FC", "size")).reset_index()
del _m
print(pd.DataFrame({"cell": CELL, "bp": {r: len(s) for r, s in SEQ.items()},
                    "SNVs": {r: len(s) for r, s in SNV0.items()}}).to_string())

In [ ]:
from pymemesuite.common import Sequence, MotifFile
from pymemesuite.fimo import FIMO
from pyfaidx import Fasta

with MotifFile(C["meme_path"]) as _mf:
    _all = list(_mf)
    BACKGROUND = _mf.background
MOTIF = {}
for m in _all:
    MOTIF.setdefault(m.name.decode(), m)           # first matrix with the name, as everywhere else
FIMO_SCAN = FIMO(both_strands=True, threshold=C["fimo_p"])
BASE_IDX = {b: i for i, b in enumerate("ACGT")}


def freq(name):
    f = np.asarray(MOTIF[name].frequencies, float)
    return f / f.sum(1, keepdims=True)


def log_odds(name, strand):
    f = freq(name) if strand == "+" else freq(name)[::-1, [3, 2, 1, 0]]
    return np.log2(np.clip(f, 1e-9, 1.0) / 0.25)


def site_score(seq, name, strand):
    lo = log_odds(name, strand)
    return float(sum(lo[j, BASE_IDX[b]] for j, b in enumerate(seq)))


def scan_many(name, seqs, batch=500):
    out = [[] for _ in seqs]
    for i0 in range(0, len(seqs), batch):
        chunk = [Sequence(s, name=str(i0 + k).encode()) for k, s in enumerate(seqs[i0:i0 + batch])]
        for h in FIMO_SCAN.score_motif(MOTIF[name], chunk, BACKGROUND).matched_elements:
            a, b = sorted((h.start, h.stop))
            out[int(h.source.name.decode())].append((a - 1, b, h.strand, h.pvalue))
    return out


LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)
CHROMS = [c for c in [f"chr{i}" for i in range(1, 23)] + ["chrX"] if c in GENOME]
CHROM_LEN = {c: len(GENOME[c]) for c in CHROMS}


def locate(seq, hint):
    # a 40 bp anchor from each end must land at the right distance; <= 2% mismatches allowed
    n, A = len(seq), 40
    order = ([f"chr{hint}".replace("chrchr", "chr")] if hint else []) + CHROMS
    for c in dict.fromkeys(order):
        if c not in GENOME:
            continue
        chrom = GENOME[c][:].seq
        for strand, q in (("+", seq), ("-", revcomp(seq))):
            i = chrom.find(q[:A])
            while i >= 0:
                if chrom[i + n - A:i + n] == q[-A:]:
                    mm = sum(a != b for a, b in zip(chrom[i:i + n], q))
                    if mm <= 0.02 * n:
                        return c, i, strand, mm
                i = chrom.find(q[:A], i + 1)
    return None


# every region in genome orientation: GSEQ, its SNVs (local position, ref/alt on the + strand), its interval
LOC = {}
for r in sorted(SEQ):
    loc = locate(SEQ[r], HINT[r])
    if not loc:
        print(f"region {r}: not found in hg38 -- skipped")
        continue
    chrom, g0, strand, mm = loc
    snv = SNV0[r].copy()
    if strand == "-":
        gseq = revcomp(SEQ[r])
        snv["local"] = len(SEQ[r]) - 1 - snv["local"]
        snv["ref"], snv["alt"] = snv["ref"].str.translate(_RC), snv["alt"].str.translate(_RC)
    else:
        gseq = SEQ[r]
    snv = snv.sort_values(["local", "alt"]).reset_index(drop=True)
    assert all(gseq[p] == b for p, b in zip(snv["local"], snv["ref"]))
    LOC[r] = dict(chrom=chrom, g0=g0, strand=strand, mm=mm, gseq=gseq, snv=snv)
    print(f"region {r:2d} ({CELL[r]}): {chrom}:{g0}-{g0 + len(gseq)} ({strand}), {mm} mismatches to hg38, {len(snv)} SNVs")
REGION_IVS = {r: (L["chrom"], L["g0"], L["g0"] + len(L["gseq"])) for r, L in LOC.items()}

## 1.2 Every JASPAR hit with SNVs inside, and ΔPWM, per region

Same scan as the per-region notebooks (FIMO, `fimo_p` from the config, both strands); a hit is kept when at least one
measured SNV lies inside it. Hit ids are per region, in FIMO-p order. Saved to `fcav_pool/regions/r<k>/`.

In [ ]:
RL = sorted(LOC)
_raw = {r: [] for r in RL}
for name in MOTIF:                                  # one FIMO call per motif over all regions (p-values do not
    for r, hs in zip(RL, scan_many(name, [LOC[r]["gseq"] for r in RL])):   # depend on the other sequences)
        _raw[r] += [(name, a, b, st, p) for a, b, st, p in hs]
HITS, PAIR = {}, {}
for r in RL:
    gseq, snv = LOC[r]["gseq"], LOC[r]["snv"]
    loc_ = snv["local"].values
    H = pd.DataFrame([{"motif": nm, "start": a, "end": b, "strand": st, "p": float(p), "pwm": site_score(gseq[a:b], nm, st)}
                      for nm, a, b, st, p in _raw[r] if ((loc_ >= a) & (loc_ < b)).any()],
                     columns=["motif", "start", "end", "strand", "p", "pwm"])
    H = H.sort_values(["p", "start", "motif"], kind="mergesort").reset_index(drop=True)
    H["hit_id"] = np.arange(len(H))
    rows = []
    for h in H.itertuples():
        lo = log_odds(h.motif, h.strand)
        for i in np.flatnonzero((loc_ >= h.start) & (loc_ < h.end)):
            k = int(loc_[i]) - h.start
            rows.append((i, h.hit_id, lo[k, BASE_IDX[snv.at[i, "alt"]]] - lo[k, BASE_IDX[snv.at[i, "ref"]]]))
    HITS[r], PAIR[r] = H, pd.DataFrame(rows, columns=["snv", "hit_id", "dpwm"])
    d = f"{POOL}/regions/r{r}"
    os.makedirs(d, exist_ok=True)
    snv.to_csv(f"{d}/snvs.csv", index=False)
    H.to_csv(f"{d}/hits.csv", index=False)
    PAIR[r].to_csv(f"{d}/snv_hit_dpwm.csv", index=False)
    print(f"region {r:2d}: {len(H)} hits of {H['motif'].nunique()} motifs contain SNVs; "
          f"{PAIR[r]['snv'].nunique()} of {len(snv)} SNVs inside at least one hit")
ALL_MOTIFS = sorted(set().union(*(set(H["motif"]) for H in HITS.values())))
print(f"\n{len(ALL_MOTIFS)} motifs with a hit in at least one region")

## 1.3 ENCODE, and the sites already built

A motif gets a direction when an ENCODE TF ChIP-seq target in HepG2 or K562 has its name (ignoring case); heterodimer
matrices (`A::B`) are skipped. Sites built earlier for regions 8 / 0 / 20 are copied, and every matched pair whose
bound or unbound window overlaps **any** MPRA region is dropped (the pair, so the matching stays intact).

In [ ]:
import requests

ENC, HDR = "https://www.encodeproject.org", {"accept": "application/json"}
ENC_DIR = f"{WORK_DIR}/encode"                      # shared download cache with the other notebooks
os.makedirs(ENC_DIR, exist_ok=True)
PEAK_COLS = ["chrom", "start", "end", "name", "score", "strand", "signal", "p", "q", "summit"]
PRIORITY = ["conservative IDR thresholded peaks", "optimal IDR thresholded peaks",
            "IDR thresholded peaks", "replicated peaks", "pseudoreplicated peaks"]


def enc_search(**params):
    r = requests.get(f"{ENC}/search/", params={**params, "format": "json", "limit": "all"},
                     headers=HDR, timeout=600)
    if r.status_code == 404:
        return []
    r.raise_for_status()
    return r.json()["@graph"]


def narrowpeak_files(assay_title, cell, target=None):
    q = {"type": "Experiment", "assay_title": assay_title, "status": "released", "biosample_ontology.term_name": cell}
    if target:
        q["target.label"] = target
    rows = []
    for e in enc_search(**q):
        for f in enc_search(type="File", dataset=e["@id"], file_format="bed", assembly="GRCh38", status="released"):
            if str(f.get("file_type", "")).startswith("bed narrowPeak"):
                rows.append({"experiment": e["accession"], "accession": f["accession"],
                             "output_type": f.get("output_type", ""),
                             "preferred_default": bool(f.get("preferred_default", False)), "href": f["href"]})
    return pd.DataFrame(rows)


def fetch_bed(row):
    path = f"{ENC_DIR}/{row['accession']}.bed.gz"
    if not os.path.exists(path):
        with requests.get(ENC + row["href"], stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
        os.rename(path + ".part", path)
    df = pd.read_csv(path, sep="\t", header=None, compression="gzip").iloc[:, :10]
    df.columns = PEAK_COLS[:df.shape[1]]
    return df


def cell_targets(cell):
    exps = enc_search(type="Experiment", assay_title="TF ChIP-seq", status="released",
                      **{"biosample_ontology.term_name": cell}, field="target.label")
    return {e["target"]["label"].upper(): e["target"]["label"] for e in exps if "target" in e}


def pick_per_experiment(files):
    keep = []
    for _, g in files.groupby("experiment"):
        if g["preferred_default"].any():
            keep.append(g[g["preferred_default"]].iloc[0])
        else:
            rank = g["output_type"].map(lambda t: PRIORITY.index(t) if t in PRIORITY else 99)
            keep.append(g.loc[rank.idxmin()])
    return pd.DataFrame(keep)


def strict_peaks(files):
    # bound: top BOUND_TOP by signal within each experiment; exclusion: every peak file of the TF
    bound = []
    for _, r in pick_per_experiment(files).iterrows():
        d = fetch_bed(r)
        bound.append(d[d["signal"] >= d["signal"].quantile(1 - BOUND_TOP)])
    excl = pd.concat([fetch_bed(r) for _, r in files.iterrows()], ignore_index=True)
    return pd.concat(bound, ignore_index=True), excl

In [ ]:
def overlaps_region(chrom, g_start):
    # the 600 bp window of a site starting at g_start overlaps some MPRA region
    ws = g_start - MOFF
    return any(chrom == c and ws < e and ws + WLEN > s for c, s, e in REGION_IVS.values())


# ---- reuse: the first earlier copy that was built, minus every pair overlapping any region
n_cp, n_drop = 0, 0
for m in ALL_MOTIFS:
    dst = f"{POOL}/sites/{m}.json"
    if "::" in m or os.path.exists(dst):
        continue
    cands = [f"{d}/{m}.json" for d in SHARE if os.path.exists(f"{d}/{m}.json")]
    if not cands:
        continue
    Js = [json.load(open(p)) for p in cands]
    J = next((j for j in Js if j["status"] == "built"), Js[0])
    rec = J["records"]
    if rec:
        nb = sum(r["cls"] == 1 for r in rec)
        assert len(rec) == 2 * nb and all(r["cls"] == 1 for r in rec[:nb])    # bound[i] is paired with unbound[i]
        keep = [i for i in range(nb) if not (overlaps_region(rec[i]["chrom"], rec[i]["g_start"])
                                              or overlaps_region(rec[nb + i]["chrom"], rec[nb + i]["g_start"]))]
        n_drop += nb - len(keep)
        J["records"] = [rec[i] for i in keep] + [rec[nb + i] for i in keep]
        if len(keep) < 50:
            J["status"], J["records"] = "too few sites after dropping MPRA-region overlaps", []
    json.dump(J, open(dst, "w"))
    n_cp += 1
print(f"reused {n_cp} TFs' sites from regions 8/0/20; {n_drop} pairs dropped for overlapping an MPRA region")

LABEL = {c: cell_targets(c) for c in CELLS}
DN_FILES = {}
for c in CELLS:
    f = narrowpeak_files("DNase-seq", c)
    DN_FILES[c] = f[f["preferred_default"]] if f["preferred_default"].any() else f
    assert len(DN_FILES[c]), f"no DNase-seq in {c}"
SOURCE = {}                                         # motif -> {cell: ChIP files}, or the reason it has none
for m in ALL_MOTIFS:
    if "::" in m:
        SOURCE[m] = "dimer matrix"
    elif os.path.exists(f"{POOL}/sites/{m}.json"):
        SOURCE[m] = "cached"
    else:
        fb = {c: narrowpeak_files("TF ChIP-seq", c, LABEL[c][m.upper()]) for c in CELLS if m.upper() in LABEL[c]}
        fb = {c: f for c, f in fb.items() if len(f)}
        SOURCE[m] = fb if fb else f"no ChIP-seq in {' or '.join(CELLS)}"
show = pd.Series({m: (v if isinstance(v, str) else "to build: " + " + ".join(v)) for m, v in SOURCE.items()})
print("  ".join(f"{c}: {len(LABEL[c])} ChIP-seq targets" for c in CELLS))
print(show.value_counts().to_string())

## 1.4 Sites for the new TFs

As in the per-region notebooks. Bound: the best hit (p < `fimo_p`) within `peak_half` of a strict peak's summit, its
centre ≤ `summit_dist` from the summit. Unbound: hits in that cell's DNase peaks ≥ `UNBOUND_MIN_DIST` from every peak
of the TF in every cell with data. Windows of 600 bp with the motif at bp 300, spaced, **none overlapping any MPRA
region**. Matched on PWM, GC and CpG o/e; at most `N_PER_CLASS` pairs. Resumable: a TF already in `sites/` is skipped.

In [ ]:
class Intervals:
    def __init__(self, df):
        self.d = {}
        for c, g in df.groupby("chrom"):
            o = np.argsort(g["start"].values)
            s, e = g["start"].values[o], g["end"].values[o]
            new = np.r_[True, s[1:] > np.maximum.accumulate(e)[:-1]]
            idx = np.flatnonzero(new)
            self.d[c] = (s[idx], np.maximum.reduceat(e, idx))

    def dist(self, chrom, pos):
        if chrom not in self.d:
            return np.inf
        st, en = self.d[chrom]
        i = np.searchsorted(st, pos, side="right") - 1
        d = np.inf
        if i >= 0:
            d = 0 if pos < en[i] else pos - en[i] + 1
        if i + 1 < len(st):
            d = min(d, st[i + 1] - pos)
        return d


def gc_frac(s):
    return (s.count("G") + s.count("C")) / len(s)


def cpg_oe(s):
    return s.count("CG") * len(s) / max(s.count("C") * s.count("G"), 1)


def site_record(m, cls, chrom, g_start, g_end, strand, p, signal=np.nan):
    ws = g_start - MOFF
    if ws < 0 or ws + WLEN > CHROM_LEN[chrom]:
        return None
    if overlaps_region(chrom, g_start):              # no window may overlap any MPRA region
        return None
    win = GENOME[chrom][ws:ws + WLEN].seq
    if "N" in win:
        return None
    return {"cls": cls, "chrom": chrom, "g_start": int(g_start), "strand": strand, "p": float(p), "signal": float(signal),
            "pwm": site_score(win[MOFF:MOFF + g_end - g_start], m, strand), "window": win}


def spaced(recs):
    kept, occ = [], set()
    for r in sorted(recs, key=lambda r: r["p"]):
        b = (r["chrom"], r["g_start"] // WLEN)
        if any((b[0], b[1] + k) in occ for k in (-1, 0, 1)):
            continue
        kept.append(r)
        occ.add(b)
    return kept


def match_pwm(bound, unbound, rng):
    # joint quantile bins of MATCH_VARS over the pooled sites; one bound to one unbound within each bin
    rec = bound + unbound
    if not bound or not unbound:
        return [], []
    for r in rec:
        r["gc"], r["cpg_oe"] = gc_frac(r["window"]), cpg_oe(r["window"])
    P = pd.DataFrame({v: [r[v] for r in rec] for v in MATCH_VARS})
    P["cls"] = [1] * len(bound) + [0] * len(unbound)
    keys = [pd.qcut(P[v].rank(method="first"), N_MATCH_BINS, labels=False) for v in MATCH_VARS]
    pairs = []
    for _, g in P.groupby(keys):
        b = rng.permutation(np.array(g.index[g["cls"] == 1]))
        u = rng.permutation(np.array(g.index[g["cls"] == 0]))
        k = min(len(b), len(u))
        pairs += list(zip(b[:k], u[:k]))
    pairs = [pairs[i] for i in rng.permutation(len(pairs))[:N_PER_CLASS]]
    return [rec[i] for i, _ in pairs], [rec[j] for _, j in pairs]


def spans(m):
    w = len(freq(m))
    return w, MOFF // TOK, -(-(MOFF + w) // TOK)


def build(m, files_by_cell):
    t0 = time.time()
    w = len(freq(m))
    peaks = {c: strict_peaks(f) for c, f in files_by_cell.items()}
    excl = Intervals(pd.concat([e for _, e in peaks.values()], ignore_index=True))   # every peak, every cell
    bound, unbound, n_pk = [], [], {}
    for c, (bpk, _) in peaks.items():
        pk = bpk[bpk["chrom"].isin(CHROMS)].copy()
        pk["abs_summit"] = np.where(pk["summit"] >= 0, pk["start"] + pk["summit"], (pk["start"] + pk["end"]) // 2)
        pk = pk.sort_values("signal", ascending=False).drop_duplicates(["chrom", "abs_summit"])
        pk = pk[(pk["abs_summit"] >= C["peak_half"]) & (pk["abs_summit"] + C["peak_half"] <= pk["chrom"].map(CHROM_LEN))]
        n_pk[c] = len(pk)
        chroms, summits, sig = pk["chrom"].tolist(), pk["abs_summit"].astype(int).tolist(), pk["signal"].tolist()
        seqs = [GENOME[ch][s - C["peak_half"]:s + C["peak_half"]].seq for ch, s in zip(chroms, summits)]
        for ch, s, sg, hits in zip(chroms, summits, sig, scan_many(m, seqs)):
            if not hits:
                continue
            a, b, st, p = min(hits, key=lambda h: h[3])
            if abs((a + b) / 2 - C["peak_half"]) > C["summit_dist"] or b - a != w:
                continue
            r = site_record(m, 1, ch, s - C["peak_half"] + a, s - C["peak_half"] + b, st, p, sg)
            if r:
                r["cell"] = c
                bound.append(r)
        want, got = UNBOUND_POOL * N_PER_CLASS // len(peaks), 0
        for i0 in range(0, len(DHS[c]), 2000):
            if got >= want:
                break
            chunk = DHS[c][i0:i0 + 2000]
            for (ch, s, e), hits in zip(chunk, scan_many(m, [GENOME[ch][s:e].seq for ch, s, e in chunk])):
                for a, b, st, p in hits:
                    if b - a != w or excl.dist(ch, s + (a + b) // 2) < UNBOUND_MIN_DIST:
                        continue
                    r = site_record(m, 0, ch, s + a, s + b, st, p)
                    if r:
                        r["cell"] = c
                        unbound.append(r)
                        got += 1
    bound, unbound = spaced(bound), spaced(unbound)
    rng = np.random.default_rng(C["seed"])
    nb, nu = len(bound), len(unbound)
    bound, unbound = match_pwm(bound, unbound, rng)
    rec = bound + unbound
    for r in rec:
        r["gc"], r["cpg_oe"] = gc_frac(r["window"]), cpg_oe(r["window"])
    w, K0, K1 = spans(m)
    ok = min(len(bound), len(unbound)) >= 50
    json.dump({"tf": m, "status": "built" if ok else f"too few sites (bound {nb}, unbound {nu})", "width": w,
               "K0": K0, "K1": K1, "pool_bound": nb, "pool_unbound": nu, "records": rec if ok else []},
              open(f"{POOL}/sites/{m}.json", "w"))
    by_cell = "  ".join(f"{c} {sum(r['cell'] == c for r in bound)}/{sum(r['cell'] == c for r in unbound)}" for c in CELLS)
    print(f"{m:10s} strict peaks {'+'.join(str(n_pk.get(c, 0)) for c in CELLS):>13s}  pool bound {nb:5d} unbound {nu:5d}"
          f" -> matched {len(bound):5d} per class (bound/unbound by cell: {by_cell})"
          f"{'' if ok else '  TOO FEW'}  [{time.time() - t0:.0f}s]", flush=True)


todo = [m for m, v in SOURCE.items() if not isinstance(v, str)]
print(f"{len(CHROMS)} chromosomes; {len(todo)} TFs to build")
if todo:
    DHS = {}
    for c in CELLS:
        _d = pd.concat([fetch_bed(r) for _, r in DN_FILES[c].iterrows()], ignore_index=True)
        _d = _d[_d["chrom"].isin(CHROMS)].sample(frac=1, random_state=C["seed"])
        DHS[c] = list(zip(_d["chrom"], _d["start"].astype(int), _d["end"].astype(int)))
        print(f"{c} DNase peaks: {len(DHS[c]):,}")
    del _d
t_all = time.time()
for k, m in enumerate(todo):
    if os.path.exists(f"{POOL}/sites/{m}.json"):     # built in an earlier, interrupted session
        continue
    print(f"[{k + 1}/{len(todo)}] ", end="")
    build(m, SOURCE[m])
print(f"done [{(time.time() - t_all) / 60:.1f} min]")

## 1.5 MPRA windows for every hit of a TF with a direction, every region

600 bp around each hit, genome bases outside the region and the MPRA's own bases inside it.

In [ ]:
STATUS = {m: (json.load(open(f"{POOL}/sites/{m}.json"))["status"] if os.path.exists(f"{POOL}/sites/{m}.json") else "not built")
          if not isinstance(v, str) or v == "cached" else v for m, v in SOURCE.items()}
HAVE = sorted(m for m, s in STATUS.items() if s == "built")
MW, rows = [], []
for r in RL:
    L = LOC[r]
    chrom, g0, gseq = L["chrom"], L["g0"], L["gseq"]
    H = HITS[r][HITS[r]["motif"].isin(HAVE)]
    for h in H.itertuples():
        w, K0, K1 = spans(h.motif)
        assert h.end - h.start == w
        ws = g0 + h.start - MOFF
        win = list(GENOME[chrom][ws:ws + WLEN].seq)
        for k in range(max(0, g0 - ws), min(WLEN, g0 + len(gseq) - ws)):
            win[k] = gseq[ws - g0 + k]                   # the MPRA bases inside the region
        win = "".join(win)
        assert len(win) == WLEN and "N" not in win and win[MOFF:MOFF + w] == gseq[h.start:h.end]
        MW.append({"region": int(r), "hit_id": int(h.hit_id), "motif": h.motif, "start": int(h.start), "end": int(h.end),
                   "strand": h.strand, "p": float(h.p), "pwm": float(h.pwm), "K0": K0, "K1": K1, "window": win})
    cov = PAIR[r][PAIR[r]["hit_id"].isin(H["hit_id"])]["snv"].nunique()
    rows.append({"region": r, "cell": CELL[r], "hits": len(HITS[r]), "hits with a direction": len(H),
                 "TFs": H["motif"].nunique(), "SNVs": len(L["snv"]), "SNVs in a hit with a direction": cov})
INFO = {str(r): {"cell": CELL[r], "chrom": LOC[r]["chrom"], "start": int(LOC[r]["g0"]),
                 "end": int(LOC[r]["g0"] + len(LOC[r]["gseq"])), "strand": LOC[r]["strand"], "mismatches": int(LOC[r]["mm"])}
        for r in RL}
json.dump({"hits": MW, "status": STATUS, "regions": INFO}, open(f"{POOL}/mpra_windows.json", "w"), default=float)
print(pd.DataFrame(rows).set_index("region").to_string())
print(f"\n{len(HAVE)} TFs with a direction; {len(MW)} hit windows over {len(RL)} regions")
print("\nmotifs without a direction:")
print(pd.Series({m: s for m, s in STATUS.items() if s != "built"}).value_counts().to_string())

In [ ]:
from google.colab import runtime
runtime.unassign()

---
# Section 2 — embed and fit, five models *(GPU runtime)*

## 2.1 Settings, models and layers

The readout is the one of the cross-model notebooks: the mean hidden state over the tokens that touch the motif's
bases; for HyenaDNA (causal) also the reverse-complement pass, concatenated. NT-v2 500M with this readout is exactly
the legacy readout (checked there). `FIT_LAYERS` are the layers used before; they were chosen by held-out ChIP AUC,
not by the MPRA.

In [ ]:
!pip install -q -U "transformers==4.57.6" "tokenizers==0.22.0" "huggingface_hub==0.34.0" "safetensors==0.4.5"
!pip install -q scikit-learn scipy pandas
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, gc, sys
import numpy as np
import torch
import torch.nn.functional as F
from contextlib import contextmanager
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, AutoConfig, AutoModelForMaskedLM, AutoModelForCausalLM

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL = f"{WORK_DIR}/fcav_pool"
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
BUILT = sorted(m for m, s in MWJ["status"].items() if s == "built")

REGISTRY = {
    "nt500m":   dict(hf="InstaDeepAI/nucleotide-transformer-v2-500m-multi-species", tok_map=("kmer", 6), causal=False, batch=32),
    "dnabert2": dict(hf="zhihan1996/DNABERT-2-117M",                                tok_map=("offsets",), causal=False, batch=32, out="dnabert2"),
    "genalm":   dict(hf="AIRI-Institute/gena-lm-bert-large-t2t",                    tok_map=("offsets",), causal=False, batch=32),
    "hyenadna": dict(hf="LongSafari/hyenadna-large-1m-seqlen-hf",                   tok_map=("char",), causal=True, batch=32, rc_mode="concat"),
    "nt2p5b":   dict(hf="InstaDeepAI/nucleotide-transformer-2.5b-multi-species",    tok_map=("kmer", 6), causal=False, batch=16),
}
MODELS = ["nt500m", "dnabert2", "genalm", "hyenadna", "nt2p5b"]   # cheapest first; cut this list to run fewer per session
FIT_LAYERS = {"nt500m": [24], "dnabert2": [4], "genalm": [8, 11], "hyenadna": [3], "nt2p5b": [14]}   # the model's own layer ids
BASE_WEIGHTED = False     # equal weight to every token that touches the motif (= the legacy readout for NT)
BATCH = 32
WLEN, MOFF = C["window_len"], C["motif_offset"]
DEV = torch.device("cuda")
COMP = str.maketrans("ACGTN", "TGCAN")
rc = lambda s: s.translate(COMP)[::-1]

HBY = {}
for h in MWJ["hits"]:
    HBY.setdefault(h["motif"], []).append(h)
ORDER = sorted(BUILT, key=lambda m: (-len(HBY.get(m, [])), m))     # most hits first
print(f"{len(BUILT)} TFs with sites, {len(MWJ['hits'])} hit windows in {len(MWJ['regions'])} regions; models {MODELS}")

## 2.2 Tokens, models, readout (copied from the cross-model notebook)

The only change: `hidden_stack` keeps just the layers in `FIT_LAYERS`.

In [ ]:
def load_tokenizer(tag):
    spec = REGISTRY[tag]
    tokz = AutoTokenizer.from_pretrained(spec["hf"], trust_remote_code=True)
    tokz.padding_side = "right"          # several of these default to left (HyenaDNA)
    if tokz.pad_token is None:
        tokz.pad_token = tokz.eos_token if tokz.eos_token is not None else tokz.unk_token
    return tokz


def token_bases(spec, n):
    # (n, 2) base interval of each sequence token, for the arithmetic maps
    kind = spec["tok_map"][0]
    if kind == "kmer":
        k = spec["tok_map"][1]
        return np.stack([np.arange(n) * k, (np.arange(n) + 1) * k], 1)
    if kind == "char":
        return np.stack([np.arange(n), np.arange(n) + 1], 1)
    raise RuntimeError("offsets are resolved by the tokenizer, not here")


def encode(tokz, spec, seqs):
    # input_ids (B, T), attention mask (B, T), and per sequence: the positions of its sequence tokens in input_ids
    # and the base interval each one covers
    kind = spec["tok_map"][0]
    if kind == "offsets":
        assert tokz.is_fast, f"{spec['hf']} has no fast tokenizer, so offset mapping is unavailable"
        enc = tokz(seqs, return_tensors="pt", padding=True, return_offsets_mapping=True)
    else:
        enc = tokz(seqs, return_tensors="pt", padding=True)
    ids = enc["input_ids"]
    am = enc.get("attention_mask")
    if am is None:
        am = (ids != tokz.pad_token_id).long()
    amn = am.numpy()
    pos, spans = [], []
    if kind == "offsets":
        om = enc["offset_mapping"].numpy()
        for b in range(len(seqs)):
            keep = (om[b, :, 1] > om[b, :, 0]) & (amn[b] == 1)
            pos.append(np.flatnonzero(keep))
            spans.append(om[b, keep])
    else:
        for b in range(len(seqs)):
            sp = np.array(tokz.get_special_tokens_mask(ids[b].tolist(), already_has_special_tokens=True))
            p = np.flatnonzero((sp == 0) & (amn[b] == 1))
            pos.append(p)
            spans.append(token_bases(spec, len(p)))
        assert len({len(p) for p in pos}) == 1, f"{spec['hf']}: windows of equal length gave different token counts"
    return ids, am, pos, spans


def span_weights(bases, lo, hi):
    # per-token weight for the base interval [lo, hi)
    ov = np.minimum(bases[:, 1], hi) - np.maximum(bases[:, 0], lo)
    ov = np.clip(ov, 0, None).astype(np.float32)
    if not BASE_WEIGHTED:
        ov = (ov > 0).astype(np.float32)
    assert ov.sum() > 0, f"no token covers bases [{lo}, {hi})"
    return ov / ov.sum()


def load_model(tag):
    spec = REGISTRY[tag]
    # the class the checkpoint ships its own code for (NT-v2 has AutoModelForMaskedLM but no AutoModel)
    amap = getattr(AutoConfig.from_pretrained(spec["hf"], trust_remote_code=True), "auto_map", None) or {}
    cls = next((c for k, c in (("AutoModel", AutoModel), ("AutoModelForMaskedLM", AutoModelForMaskedLM),
                               ("AutoModelForCausalLM", AutoModelForCausalLM)) if k in amap), AutoModel)
    mdl = cls.from_pretrained(spec["hf"], trust_remote_code=True).to(DEV).eval()
    # DNABERT-2's triton flash-attention kernel does not compile on current triton; its code falls back to PyTorch
    for name in {type(m).__module__ for m in mdl.modules()}:
        mod = sys.modules.get(name)
        if mod is not None and getattr(mod, "flash_attn_qkvpacked_func", None) is not None:
            mod.flash_attn_qkvpacked_func = None
            print(f"{tag}: triton flash-attention disabled in {name.split('.')[-1]}; using PyTorch attention")
    if spec.get("out") == "dnabert2" and getattr(mdl, "pooler", None) is not None:
        mdl.pooler = None
    return mdl


POS, DEPTH, LOADED = {}, {}, {}


@torch.no_grad()
def hidden_stack(tag, model, ids, am):
    # one (layers, T_j, H) float tensor per sequence; only the layers in POS[tag] once prepare() has set it
    spec = REGISTRY[tag]
    if spec.get("out") == "dnabert2":
        # DNABERT-2 returns every encoder layer unpadded; one sequence at a time keeps the token order
        out = []
        for j in range(ids.shape[0]):
            t = ids[j][am[j].bool()][None].to(DEV)
            layers, _ = model(t, attention_mask=torch.ones_like(t), output_all_encoded_layers=True)
            S = torch.stack([x.reshape(t.shape[1], -1) for x in layers], 0).float()
            out.append(S[POS[tag]] if tag in POS else S)
        return out
    kw = {} if spec["causal"] else {"attention_mask": am.to(DEV)}
    hs = model(input_ids=ids.to(DEV), output_hidden_states=True, **kw).hidden_states
    if tag in POS:
        hs = [hs[i] for i in POS[tag]]
    S = torch.stack(hs, 1).float()
    return [S[j] for j in range(S.shape[0])]


PROBE = [h["window"] for h in MWJ["hits"][:4]]


def prepare(tag):
    # load the model; from one forward pass: number of layers, token tiling, and where FIT_LAYERS sit in the stack
    spec = REGISTRY[tag]
    mdl = load_model(tag)
    tokz = LOADED.get(tag, {}).get("tokz") or load_tokenizer(tag)
    LOADED[tag] = dict(spec=spec, tokz=tokz)
    POS.pop(tag, None)
    ids, am, pos, spans = encode(tokz, spec, PROBE[:2])
    S = hidden_stack(tag, mdl, ids, am)
    for j, Sj in enumerate(S):
        assert Sj.ndim == 3 and int(pos[j].max()) < Sj.shape[1]
        b = spans[j]
        assert b[0, 0] == 0 and b[-1, 1] == WLEN and (b[1:, 0] == b[:-1, 1]).all(), f"{tag}: tokens do not tile"
    nl = S[0].shape[0]
    lids = list(range(1, nl + 1)) if spec.get("out") == "dnabert2" else list(range(nl))   # DNABERT-2: no embedding layer
    DEPTH[tag] = lids[-1]
    miss = [l for l in FIT_LAYERS[tag] if l not in lids]
    assert not miss, f"{tag}: layers {miss} do not exist (ids {lids[0]}..{lids[-1]})"
    POS[tag] = [lids.index(l) for l in FIT_LAYERS[tag]]
    print(f"{tag}: {nl} layers (depth {DEPTH[tag]}), fitting {FIT_LAYERS[tag]}, hidden {S[0].shape[2]}, "
          f"{len(pos[0])} tokens for {WLEN} bp")
    del S
    return mdl


@torch.no_grad()
def readout(tag, model, seqs, lohi):
    # (n, layers, D): D = hidden, or 2 * hidden for a causal model (forward ‖ reverse complement)
    L = LOADED[tag]
    tokz, spec = L["tokz"], L["spec"]
    passes = [(seqs, lohi)]
    if spec["causal"]:
        passes.append(([rc(s) for s in seqs], [(WLEN - b, WLEN - a) for a, b in lohi]))
    bs = spec.get("batch", BATCH)
    parts = []
    for ss, lh in passes:
        out = None
        for i in range(0, len(ss), bs):
            chunk, lhc = ss[i:i + bs], lh[i:i + bs]
            ids, am, pos, spans = encode(tokz, spec, chunk)
            S = hidden_stack(tag, model, ids, am)
            if out is None:
                out = np.zeros((len(ss), S[0].shape[0], S[0].shape[2]), np.float16)
            for j in range(len(chunk)):
                w = torch.tensor(span_weights(spans[j], *lhc[j]), device=DEV)
                p = torch.tensor(pos[j], device=DEV)
                out[i + j] = (S[j][:, p] * w[None, :, None]).sum(1).half().cpu().numpy()
            del S
        parts.append(out)
    if len(parts) == 1:
        return parts[0]
    if spec.get("rc_mode", "concat") == "mean":
        return ((parts[0].astype(np.float32) + parts[1].astype(np.float32)) / 2).astype(np.float16)
    return np.concatenate(parts, 2)

## 2.3 The estimator (copied unchanged from the cross-model notebook)

In [ ]:
torch.backends.cuda.preferred_linalg_library("cusolver")
LAMS = torch.logspace(4, -1, 11, device=DEV)      # L2 penalties tried, strongest first (standardised features, summed log-loss)
NEWTON = 15                                       # max Newton steps per penalty
INNER = 3                                         # inner CV folds for the penalty
FAST = True       # Hessian in TF32 / Cholesky in float32, float64 fallback for the layers where that fails


@contextmanager
def _tf32(on):
    old = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = on
    try:
        yield
    finally:
        torch.backends.cuda.matmul.allow_tf32 = old


def logit_path(X, y):
    # L2 logistic regression for every layer at once and every penalty in LAMS (warm-started, strongest first).
    # X: (layers, n, d) standardised; y: (n,) 0/1. Returns W (lams, layers, d + 1) = [weights, intercept].
    nl, n, d = X.shape
    Xa = torch.cat([X, torch.ones(nl, n, 1, device=X.device)], 2)
    pen = torch.ones(d + 1, device=X.device); pen[-1] = 0                # intercept not penalised
    w = torch.zeros(nl, d + 1, device=X.device)
    w[:, -1] = torch.logit(y.mean().clamp(1e-3, 1 - 1e-3))

    def loss(w, lam):
        z = torch.einsum("lnd,ld->ln", Xa, w)
        return F.binary_cross_entropy_with_logits(z, y.expand_as(z), reduction="none").sum(1) + lam / 2 * (pen * w * w).sum(1)

    out = []
    for lam in LAMS.tolist():
        for _ in range(NEWTON):
            z = torch.einsum("lnd,ld->ln", Xa, w)
            p = torch.sigmoid(z)
            g = torch.einsum("lnd,ln->ld", Xa, p - y) + lam * pen * w
            s_ = p * (1 - p)
            with _tf32(FAST):
                H = (Xa * s_[..., None]).transpose(1, 2) @ Xa + torch.diag(lam * pen + 1e-6)
            if FAST:
                Lc, info = torch.linalg.cholesky_ex(H)
                step = torch.cholesky_solve(g[..., None], Lc)[..., 0]
                bad = (info > 0) | ~torch.isfinite(step).all(1)
                if bad.any():                                                 # exact Hessian, float64, those layers only
                    Xb = Xa[bad].double()
                    Hb = (Xb * s_[bad].double()[..., None]).transpose(1, 2) @ Xb + torch.diag(lam * pen + 1e-6).double()
                    Lb, ib = torch.linalg.cholesky_ex(Hb)
                    sb = torch.cholesky_solve(g[bad].double()[..., None], Lb)[..., 0].float()
                    sb[ib > 0] = 0
                    step[bad] = sb
                    del Xb, Hb, Lb
            else:
                Lc, info = torch.linalg.cholesky_ex(H.double())
                step = torch.cholesky_solve(g.double()[..., None], Lc)[..., 0].float()
                step[info > 0] = 0
            step = torch.where(torch.isfinite(step), step, torch.zeros_like(step))
            ts = torch.tensor([1, .5, .25, .125, 0], device=X.device)
            L = torch.stack([loss(w - t * step, lam) for t in ts])         # (steps, layers); t = 0 keeps w
            L = torch.nan_to_num(L, nan=float("inf"))
            t = ts[L.argmin(0)][:, None]
            w = w - t * step
            if (t * step).abs().max() < 1e-4:
                break
        out.append(w.clone())
    return torch.stack(out)


def bce(Xte, yte, W):
    # held-out summed log-loss of every (penalty, layer); Xte (layers, n, d), W (lams, layers, d + 1)
    z = torch.einsum("lnd,kld->kln", Xte, W[..., :-1]) + W[..., -1:]
    return F.binary_cross_entropy_with_logits(z, yte.expand_as(z), reduction="none").sum(2)


def fit_logit(X, y, seed):
    # penalty per layer by INNER-fold CV log-loss on these sites, then the model on all of them at that penalty
    ll = torch.zeros(len(LAMS), X.shape[0], device=X.device)
    yn = y.cpu().numpy()
    for itr, ite in StratifiedKFold(INNER, shuffle=True, random_state=seed).split(np.zeros(len(yn)), yn):
        itr, ite = torch.tensor(itr, device=X.device), torch.tensor(ite, device=X.device)
        ll += bce(X[:, ite], y[ite], logit_path(X[:, itr], y[itr]))
    k = torch.nan_to_num(ll, nan=float("inf")).argmin(0)
    Wp = logit_path(X, y)
    return Wp[k, torch.arange(X.shape[0])], k.cpu().numpy(), Wp


def fit_one(X, y, seed, Hh):
    # 5 outer folds, penalty by inner CV; out-of-fold AUC per layer; each hit's logit averaged over the 5 fold models
    folds = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y))
    Xg = torch.tensor(X, dtype=torch.float32, device=DEV).transpose(0, 1).contiguous()
    yg = torch.tensor(y, dtype=torch.float32, device=DEV)
    Hg = torch.tensor(Hh, dtype=torch.float32, device=DEV).transpose(0, 1).contiguous()
    nl = Xg.shape[0]
    oof = np.zeros((len(y), nl), np.float32)
    dk, qk, kk = [], [], []
    for f, (tr, te) in enumerate(folds):
        mu, sd = Xg[:, tr].mean(1, keepdim=True), Xg[:, tr].std(1, keepdim=True) + 1e-6
        W, ki, _ = fit_logit((Xg[:, tr] - mu) / sd, yg[tr], seed + f)
        kk.append(ki)
        oof[te] = (torch.einsum("lnd,ld->ln", (Xg[:, te] - mu) / sd, W[:, :-1]) + W[:, -1:]).cpu().numpy().T
        dh = (torch.einsum("lnd,ld->ln", (Hg - mu) / sd, W[:, :-1]) + W[:, -1:]).cpu().numpy().T
        dk.append(dh); qk.append(1 / (1 + np.exp(-dh)))
    assert np.isfinite(oof).all(), "non-finite out-of-fold logits"
    auc = np.array([roc_auc_score(y, oof[:, li]) for li in range(nl)])
    mu, sd = Xg.mean(1, keepdim=True), Xg.std(1, keepdim=True) + 1e-6
    Wp = logit_path((Xg - mu) / sd, yg)
    kall = np.median(np.array(kk), 0).round().astype(int)
    W = Wp[torch.tensor(kall, device=DEV), torch.arange(nl, device=DEV)]
    res = dict(auc=auc, oof=oof, cls=y, w_all=W.cpu().numpy(), mu=mu[:, 0].cpu().numpy(), sd=sd[:, 0].cpu().numpy(),
               lam_all=LAMS.cpu().numpy()[kall], dec_hit=np.mean(dk, 0).astype(np.float32),
               prob_hit=np.mean(qk, 0).astype(np.float32))
    del Xg, Hg, Wp; gc.collect(); torch.cuda.empty_cache()
    return res

## 2.4 Fit every TF on every model

One model at a time, TFs with the most hits first. Each `fits/<model>/<TF>.npz` holds the out-of-fold AUC, every hit's
P(bound) (`prob_hit`, `dec_hit` = its logit) and the hits' `(hit_region, hit_id)`. Written to a temporary name and
renamed, so an interrupted write never leaves a broken file. The printed "h left" is a running estimate.

In [ ]:
t_all = time.time()
for tag in MODELS:
    OUTD = f"{POOL}/fits/{tag}"
    os.makedirs(OUTD, exist_ok=True)
    todo = [m for m in ORDER if not os.path.exists(f"{OUTD}/{m}.npz")]
    print(f"\n=== {tag}: {len(todo)} of {len(ORDER)} TFs to do", flush=True)
    if not todo:
        continue
    mdl = prepare(tag)
    t_m = time.time()
    for k, m in enumerate(todo):
        t0 = time.time()
        J = json.load(open(f"{POOL}/sites/{m}.json"))
        R = J["records"]
        w = int(J.get("width", (J["K1"] - J["K0"]) * 6))
        y = np.array([r["cls"] for r in R])
        hi = HBY[m]
        assert all(h["end"] - h["start"] == w for h in hi)
        X = readout(tag, mdl, [r["window"] for r in R], [(MOFF, MOFF + w)] * len(R))
        XH = readout(tag, mdl, [h["window"] for h in hi], [(MOFF, MOFF + w)] * len(hi))
        assert np.isfinite(X).all() and np.isfinite(XH).all(), f"{tag} {m}: a hidden state overflowed float16"
        res = fit_one(X, y, C["seed"], XH)
        tmp = f"{OUTD}/{m}.part.npz"
        np.savez(tmp, hit_region=np.array([h["region"] for h in hi]), hit_id=np.array([h["hit_id"] for h in hi]),
                 layers=np.array(FIT_LAYERS[tag]), n_layers=DEPTH[tag], model=tag, rc=REGISTRY[tag]["causal"],
                 rc_mode=REGISTRY[tag].get("rc_mode", "concat"), base_weighted=BASE_WEIGHTED, **res)
        os.replace(tmp, f"{OUTD}/{m}.npz")
        left = (time.time() - t_m) / (k + 1) * (len(todo) - k - 1) / 3600
        print(f"[{tag} {k + 1}/{len(todo)}] {m:10s} sites {len(y):5d}  hits {len(hi):3d} in "
              f"{len({h['region'] for h in hi}):2d} regions  AUC " + "  ".join(f"L{l} {a:.3f}" for l, a in zip(FIT_LAYERS[tag], res["auc"]))
              + f"  [{time.time() - t0:.0f}s; ~{left:.1f} h left for {tag}]", flush=True)
        del X, XH; gc.collect(); torch.cuda.empty_cache()
    del mdl; gc.collect(); torch.cuda.empty_cache()
print(f"\ndone [{(time.time() - t_all) / 60:.1f} min]")

In [ ]:
from google.colab import runtime
runtime.unassign()